# L6 — Datadog Agent Observability

## What You'll Learn

- How the **deployed** order/refund agents emit Agent Observability spans (this notebook creates none)
- The agent pattern captured live from the AgentCore runtimes: **Agent → LLM → Tool**
- How to drive real traffic and tour the resulting traces in the Datadog LLM Observability UI

## What is Agent Observability?

Datadog Agent Observability (formerly LLM Observability) goes beyond infrastructure
monitoring. It captures the **AI semantics** of your system:

| Span Kind | What it represents | Example (this workshop) |
|-----------|-------------------|----------|
| **Agent** | One autonomous agent invocation | An `invoke_harness` run of the Order Agent |
| **LLM** | A call to a language model | litellm → Claude on Bedrock generating a response |
| **Tool** | An external tool call the agent makes | `check_order_details` via the Gateway |

## Where these spans come from

Unlike notebooks 1–2, this notebook does **not** create spans. The order and refund
agents were deployed (L3 Notebooks 2-3) with `LLMOBS_ENABLED=true`. In the
container, `agents/observability.py` turns on LLM Observability **sidecar-routed**
(through the trace-agent's EVP proxy, not agentless) and its `LLMObsAgentHooks` opens an
**Agent** span per invocation with the **LLM** and **Tool** spans nested underneath — the
agent pattern. This notebook just drives real traffic and tours the traces those live
runtimes emit.

## Prerequisites

- **L1–L3 notebooks completed** (Harness deployed, SSM params populated)
- **Agents deployed with `LLMOBS_ENABLED=true`** — set at deploy time in L3 Notebooks 2-3
- ML app: `anycompany-customer-support`

## License Details

Refer to LICENSE file in the main folder for license details.

## Step 1: Install Dependencies

In [ ]:
%pip install --quiet boto3 --upgrade

## Step 2: Confirm How the Spans Are Produced

This notebook creates **no** spans. The Agent Observability spans come from the deployed
agents' `LLMObsAgentHooks` (`agents/observability.py`), enabled at deploy time via
`LLMOBS_ENABLED=true` and routed through the in-container trace-agent sidecar. Below we
just set `DD_SITE` (to build the trace link) and the ML-app name.

In [ ]:
import os, time, uuid, json
import boto3
from botocore.config import Config

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'
ML_APP = 'anycompany-customer-support'   # matches DD_LLMOBS_ML_APP on the deployed agents

# DD_SITE is used only to build the LLM Observability trace URL below.
DD_SITE = os.environ.get('DD_SITE', '')
if not DD_SITE:
    try:
        DD_SITE = boto3.client('ssm', region_name=REGION).get_parameter(
            Name=SSM_PREFIX + '/dd_site')['Parameter']['Value']
    except Exception:
        DD_SITE = 'datadoghq.com'

print('Agent Observability is emitted by the DEPLOYED agents (LLMOBS_ENABLED=true),')
print('sidecar-routed via the trace-agent EVP proxy — this notebook creates no spans.')
print('  ml_app:', ML_APP)
print('  site:  ', DD_SITE)

## Step 3: Load Config & Authenticate

In [ ]:
REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'

ssm = boto3.client('ssm', region_name=REGION)
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore = boto3.client('bedrock-agentcore', region_name=REGION, config=Config(read_timeout=300))

def get_ssm(name):
    try: return ssm.get_parameter(Name=name, WithDecryption=True)['Parameter']['Value']
    except: return None

cfg = {
    'harness_arn': get_ssm(f'{SSM_PREFIX}/harness_arn'),
    'cognito_client_id': get_ssm(f'{SSM_PREFIX}/cognito_client_id'),
    'user_password': get_ssm(f'{SSM_PREFIX}/user_password'),
    'model_id': get_ssm(f'{SSM_PREFIX}/model_id') or 'bedrock/us.anthropic.claude-sonnet-4-6',
}

if not cfg['harness_arn']:
    raise RuntimeError('harness_arn not in SSM. Run L3 notebooks first.')

auth_resp = cognito.initiate_auth(
    ClientId=cfg['cognito_client_id'],
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': 'gold_customer', 'PASSWORD': cfg['user_password']},
)
print('Authenticated as gold_customer')


## Step 4: Drive Live Traffic Through the Agents

Each `invoke_harness` call runs the orchestrator and specialists on the AgentCore
runtimes, both with Agent Observability enabled. We send a couple of
representative prompts (an order lookup and a refund query) to generate real traces.

Spans land under **two ML apps**:
- **Orchestrator** (`anycompany-orchestrator`) - the AWS-managed Harness exports OTLP
  using OpenTelemetry GenAI conventions, which Datadog converts into the full agent
  pattern: **`workflow`**, **`agent`**, **`tool`**, and **`llm`** spans.
- **Specialists** (`anycompany-customer-support`) - the order/refund runtimes emit
  **`llm`** spans (model call, prompt/completion, token usage) via the litellm integration.

In [ ]:
# Representative prompts that exercise the order and refund specialists.
PROMPTS = [
    'What is the status of order ORD-10001?',
    'What is your refund policy for a defective item?',
]
ACTOR_ID = 'CUST-789'


def invoke(prompt):
    session_id = str(uuid.uuid4()).upper()
    t0 = time.time()
    resp = agentcore.invoke_harness(
        harnessArn=cfg['harness_arn'],
        runtimeSessionId=session_id,
        actorId=ACTOR_ID,
        messages=[{'role': 'user', 'content': [{'text': f'[Authenticated customer: {ACTOR_ID} | Tier: gold]\n\n{prompt}'}]}],
    )
    text, tools = '', []
    for event in resp['stream']:
        if 'contentBlockDelta' in event:
            text += event['contentBlockDelta'].get('delta', {}).get('text', '')
        elif 'contentBlockStart' in event:
            tu = event['contentBlockStart'].get('start', {}).get('toolUse')
            if tu:
                tools.append(tu.get('name', 'unknown'))
    return text, tools, time.time() - t0, session_id


for p in PROMPTS:
    print(f'\n=== Prompt: {p}')
    text, tools, secs, sid = invoke(p)
    print(f'  session {sid}  ({secs:.1f}s)')
    if tools:
        print(f'  tools called: {", ".join(tools)}')
    print('  response:', (text[:300] + ('...' if len(text) > 300 else '')))

print('\nOrchestrator run -> workflow/agent/tool/llm spans (anycompany-orchestrator).')
print('Specialist runs   -> llm spans (anycompany-customer-support).')

## Step 5: Inspect the LLM Calls (model, tokens, latency)

Pull the model calls the agents just made and show the **model, token usage, and latency
per call** - the AI-specific detail, straight from the `gen_ai` spans captured in APM. The
full prompt/completion text and the Agent -> LLM -> Tool nesting live in the LLM
Observability UI (next step).


In [ ]:
# Per-LLM-call detail from the gen_ai (Bedrock ConverseStream) spans the agents emit.
from datadog_api_client import Configuration, ApiClient
from datadog_api_client.v2.api.spans_api import SpansApi
from datadog_api_client.v2.model.spans_list_request import SpansListRequest
from datadog_api_client.v2.model.spans_list_request_data import SpansListRequestData
from datadog_api_client.v2.model.spans_list_request_attributes import SpansListRequestAttributes
from datadog_api_client.v2.model.spans_list_request_page import SpansListRequestPage
from datadog_api_client.v2.model.spans_sort import SpansSort
from datetime import datetime, timedelta, timezone

_api = os.environ.get('DD_API_KEY', '')
_app = os.environ.get('DD_APP_KEY', '')
if not _api or not _app:
    _s = boto3.client('ssm', region_name=REGION)
    _api = _s.get_parameter(Name=SSM_PREFIX + '/dd_api_key', WithDecryption=True)['Parameter']['Value']
    _app = _s.get_parameter(Name=SSM_PREFIX + '/dd_app_key', WithDecryption=True)['Parameter']['Value']
_cfg = Configuration()
_cfg.api_key['apiKeyAuth'] = _api
_cfg.api_key['appKeyAuth'] = _app
_cfg.server_variables['site'] = DD_SITE

def _usage(a):
    return (a.get('custom', {}).get('gen_ai', {}) or {}).get('usage', {}) or {}

def _fetch(minutes):
    _now = datetime.now(timezone.utc)
    with ApiClient(_cfg) as client:
        body = SpansListRequest(data=SpansListRequestData(type='search_request',
            attributes=SpansListRequestAttributes(
                filter={'from': (_now - timedelta(minutes=minutes)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                        'to': _now.strftime('%Y-%m-%dT%H:%M:%SZ'), 'query': 'service:anycompany-*'},
                sort=SpansSort.TIMESTAMP_DESCENDING, page=SpansListRequestPage(limit=1000))))
        data = SpansApi(client).list_spans(body=body).get('data', [])
    return [s.get('attributes', {}) for s in data]

# Filter client-side for spans carrying gen_ai token usage (robust vs resource naming).
rows = [a for a in _fetch(30) if _usage(a).get('input_tokens') is not None]
if not rows:
    rows = [a for a in _fetch(360) if _usage(a).get('input_tokens') is not None]

def _ms(a):
    st, en = a.get('start_timestamp'), a.get('end_timestamp')
    try:
        return (en - st).total_seconds() * 1000 if (st and en) else 0.0
    except Exception:
        return 0.0

print(f'LLM calls found: {len(rows)}\n')
print(f'{"Service":<26} {"Model":<32} {"in tok":>8} {"out tok":>8} {"ms":>7}')
print('-' * 86)
tot_in = tot_out = 0
for a in rows[:25]:
    ga = a.get('custom', {}).get('gen_ai', {}) or {}
    u = ga.get('usage', {}) or {}
    model = (ga.get('request', {}) or {}).get('model', '?')
    i, o = int(u.get('input_tokens', 0) or 0), int(u.get('output_tokens', 0) or 0)
    tot_in += i; tot_out += o
    print(f'{a.get("service", "?")[:25]:<26} {str(model)[:31]:<32} {i:>8,} {o:>8,} {_ms(a):>7.0f}')
print('-' * 86)
print(f'{"TOTAL":<26} {"":<32} {tot_in:>8,} {tot_out:>8,}')
if not rows:
    print('\nNo gen_ai LLM spans found - run Step 4 (drive traffic) above, then re-run.')


## Step 6: View the Live Agent-Pattern Traces in Datadog

The deployed agents flush their spans through the sidecar automatically — there is
nothing to flush from this notebook. Give the spans a few seconds to arrive, then open
the LLM Observability UI.

In [ ]:
time.sleep(8)  # let the sidecar deliver the spans

base = f'https://app.{DD_SITE}' if DD_SITE else 'https://app.datadoghq.com'
print('Agent-pattern traces (orchestrator - full workflow/agent/tool/llm):')
print(f'  Orchestrator:     {base}/llm/traces?query=%40ml_app%3Aanycompany-orchestrator')
print()
print('Specialist model calls (llm spans - prompt, completion, tokens):')
print(f'  All specialists:  {base}/llm/traces?query=%40ml_app%3A{ML_APP}')
print(f'  Order Agent:      {base}/llm/traces?query=%40ml_app%3A{ML_APP}%20service%3Aanycompany-order-agent')
print(f'  Refund Agent:     {base}/llm/traces?query=%40ml_app%3A{ML_APP}%20service%3Aanycompany-refund-agent')
print()
print('What you will see:')
print('  Orchestrator app: workflow + agent + tool + llm spans (OTLP GenAI conventions)')
print('  Specialist apps:  llm spans (model call, prompt/completion, token usage)')

## How This Differs from APM Traces (Notebook 1)

| | APM (Notebook 1) | Agent Observability (This Notebook) |
|---|---|---|
| **Span types** | Generic HTTP, Lambda | Agent, LLM, Tool |
| **Prompts visible** | No | Yes (full input/output text) |
| **Token usage** | Aggregate (not per-call) | Per-LLM-span |
| **Model info** | Not captured | Model name, provider, parameters |
| **Where to view** | APM > Traces | AI Observability > Investigate > Traces |
| **Instrumentation** | trace-agent sidecar via `ddtrace-run` (auto) | the deployed agents' `LLMObsAgentHooks` (`agents/observability.py`) |
| **Egress** | trace-agent sidecar (`localhost:8126`) | same sidecar, EVP proxy (`localhost:8126/evp_proxy`) |

Both are emitted by the **live** runtimes and both leave the container through the same
trace-agent sidecar — APM gives infrastructure visibility, Agent Observability gives the
AI-specific view. Neither is produced by this notebook.

### Agent Observability Traces in Datadog UI

The Agent Observability traces are visible at **AI Observability > Investigate > Traces**.
Each trace shows the full agent workflow: prompts, completions, tool calls, and timing.

![Datadog LLM Traces](datadog_llm_traces.png)


## Two Instrumentation Paths in One System

The spans above come from the specialist agents. The **orchestrator** reaches Datadog a
completely different way, and the runtime is what forces the choice:

| | Specialist agents (this notebook) | Orchestrator (Harness) |
|---|---|---|
| **Runtime** | Container image we build | AWS-managed AgentCore Harness |
| **Instrumentation** | `ddtrace-run` + `LLMObsAgentHooks` | OTel: `strands.telemetry.tracer` + `opentelemetry.instrumentation.botocore` |
| **Egress** | trace-agent sidecar (`localhost:8126`) | OTLP-direct to Datadog's intake, environment variables only |
| **Configured in** | L3 notebooks 2-3 (`LLMOBS_ENABLED=true`) | L3 notebook 4 (`OTEL_*` variables) |
| **ML app** | `anycompany-customer-support` | `anycompany-orchestrator` |

The Harness has no container to modify and no sidecar to attach, so environment variables are
the only lever available. Setting `dd-otlp-source=llmobs` on its OTLP headers makes Datadog
run the GenAI conversion, so those spans populate both APM and Agent Observability from one
exporter - `gen_ai.usage.*` becomes token metrics, `gen_ai.operation.name` becomes span kind,
`gen_ai.request.*` becomes span metadata.

### Finding the orchestrator's spans

For the OTLP path `ml_app` is derived from `OTEL_SERVICE_NAME`, so the orchestrator lands in
its own application and a single delegated request spans **two ML applications**. To see the
orchestrator side, switch the ML app selector to `anycompany-orchestrator`, then:

1. Choose the **Spans** tab - Traces matches root spans only, and the root is an agent span
2. Query `@meta.span.kind:llm`

Two things that trip people up: attribute queries take an `@` prefix (`@meta.span.kind`) while
tags do not (`ml_app`), and the OTLP conversion lags **3-5 minutes** where sidecar-routed
agent spans appear within seconds. An empty view right after a run usually means you looked
too early.


## Summary

| What | How |
|---|---|
| Enabled Agent Observability | On the deployed agents via `LLMOBS_ENABLED=true` (L3 Notebooks 2-3) |
| Emitted the agent pattern | `LLMObsAgentHooks` in `agents/observability.py` — Agent span per run, nested LLM + Tool spans |
| Routed the spans | Sidecar-routed through the trace-agent EVP proxy (not agentless — keeps the AI Guard APM span on the sidecar) |
| Generated traffic | `invoke_harness` from this notebook (the runtimes produced the spans) |
| Viewed | AI Observability > Investigate > Traces, `ml_app:anycompany-customer-support` |

### Workshop Complete

You now have three levels of observability for your multi-agent system:

1. **Infrastructure** (Notebook 1) — distributed traces via APM
2. **Operational** (Notebook 2) — metrics, cost, alerting, dashboards
3. **AI-Specific** (This Notebook) — prompts, completions, agent decisions, live from the runtimes

## Cleanup

There is nothing to disable in this notebook — it enables no LLM Observability of its
own. To stop the deployed agents from emitting Agent Observability spans, redeploy them
without `LLMOBS_ENABLED=true` (re-run L3 Notebooks 2-3 with that env var removed).

Traces already sent to Datadog are retained per your plan's retention policy and expire
automatically.

In [ ]:
# === CLEANUP ===
# This notebook enables no LLM Observability of its own, so there is nothing to disable
# here. Agent Observability is turned on at DEPLOY time (LLMOBS_ENABLED=true) and off by
# redeploying the agents without it:
#
#   re-run L3 notebooks 2/3 with LLMOBS_ENABLED removed from the launch env_vars.
#
# Traces already in Datadog expire per your retention policy.
print('Nothing to clean up in this notebook - Agent Observability is a deploy-time toggle.')